In [1]:
!apt-get update -qq
!apt-get install -y -qq libgl1 libglib2.0-0

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!pip install --upgrade pip
!pip install uv
!uv pip install -U "mineru[all]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 183.3 MB/s  0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 223 packages in 3.58s
Prepared 134 packages in 41.61s
Uninstalled 62 packages in 2.18s
Installed 134 packages in 1.81s
 - aiohappyeyeballs==2.6.1
 + aiohappyeyeballs==2.6.2
 + anthropic==0.71.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + av==17.0.1
 - beautifulsoup4==4.13.5
 + beautifulsoup4==4.14.3
 + blake3==1.0.8
 + boto3==1.43.12
 + botocore==1.43.12
 - cachetools==6.2.6
 + cachetools==7.1.3
 + cbor2==6.1.1
 - certifi==2026.4.22
 + certifi==2026.5.20
 - click==8.3.3
 + click==8.4.0
 + cobble==0.1.4
 + colorlog==6.10.1
 + compressed-tensors==0.12.2
 - cryptography==43.0.3
 + cryptography==48.0.0
 - cuda-bindings==12.9.4
 + cuda-bindings==13

In [3]:
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.51.3" "tokenizers==0.21.1"
!pip install -q -U "mineru[all]"

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.11.2 requires transformers<5,>=4.56.0, but you have transformers 4.51.3 which is incompatible.


In [4]:
!mineru --help

Usage: mineru [OPTIONS]

Options:
  -v, --version                   display the version and exit
  -p, --path PATH                 local filepath or directory. support pdf,
                                  image, docx, pptx, xlsx files  [required]
  -o, --output PATH               output local directory  [required]
  --api-url TEXT                  MinerU FastAPI base URL. If omitted, mineru
                                  starts a temporary local mineru-api service.
  -m, --method [auto|txt|ocr]     the method for parsing pdf:
                                    auto: Automatically determine the method based on the file type.
                                    txt: Use text extraction method.
                                    ocr: Use OCR method for image-based PDFs.
                                  Without method specified, 'auto' will be used by default.
                                  Adapted only for the case where the backend is set to 'pipeline' and 'hybrid-*'.
  -b, --

In [18]:
!pip install -q  pymupdf

In [4]:
pdf1_paths = ["/content/drive/MyDrive/프젝랩/(1-7장)산업용기계중대재해사례집_2012년.pdf"]

In [5]:
from pathlib import Path
import shutil
import json

In [6]:
work_dir = Path("/content/drive/MyDrive/프젝랩")
input_dir = Path("/content/drive/MyDrive/프젝랩/input")
output_dir = Path("/content/drive/MyDrive/프젝랩/out")

input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

def ascii_paths(pdf_paths):
  copied = []
  for i, pdf_path in enumerate(pdf_paths, 1):
    pdf_path = Path(pdf_path)
    output_path = input_dir / f"{i:03d}.pdf"
    shutil.copy2(pdf_path, output_path)
    copied.append({
        "index": i,
        "original_path": str(pdf_path),
        "ascii_path": str(output_path),
        "original_name": pdf_path.name
    })

  manifest_path = work_dir / "manifest.json"
  with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(copied, f, ensure_ascii=False, indent=2)

  print(f"copied {len(copied)} files")
  return copied

copied_files = ascii_paths(pdf1_paths)
copied_files


copied 1 files


[{'index': 1,
  'original_path': '/content/drive/MyDrive/프젝랩/(1-7장)산업용기계중대재해사례집_2012년.pdf',
  'ascii_path': '/content/drive/MyDrive/프젝랩/input/001.pdf',
  'original_name': '(1-7장)산업용기계중대재해사례집_2012년.pdf'}]

In [7]:
import subprocess

In [24]:
import os
import json
from pathlib import Path
import shutil
import fitz

In [9]:
def get_mineru_help_text():
    try:
        r = subprocess.run(
            ["mineru", "--help"],
            capture_output=True,
            text=True,
            check=False
        )
        return (r.stdout or "") + "\n" + (r.stderr or "")
    except Exception as e:
        return str(e)

MINERU_HELP = get_mineru_help_text()
AUTO_LANG_SUPPORTED = ("auto" in MINERU_HELP and "--lang" in MINERU_HELP)

print("AUTO_LANG_SUPPORTED:", AUTO_LANG_SUPPORTED)

AUTO_LANG_SUPPORTED: True


In [29]:
def run_mineru(pdf_path, lang="korean", out_root=output_dir, method="auto"):
  pdf_path = Path(pdf_path)

  out_dir = Path(out_root) / pdf_path.stem
  out_dir.mkdir(parents=True, exist_ok=True)

  done_path = out_dir / "done.json"
  log_path = out_dir / "run_log.txt"

  live_log_path = out_dir / "run_log_live.txt"
  json_log_path = out_dir / "run_log.json"

  run_name = f"{pdf_path.stem}_{method}_{lang}_{out_dir}"

  if done_path.exists():
    print("skip", pdf_path.name)
    return out_dir, 0

  cmd = ["mineru", "-p", str(pdf_path), "-o", str(out_dir), "-m", method, "-l", lang, "-f", "false", "-t", "false"]

  print("="*10)
  print(f"Running: ")
  print(" ".join(cmd))
  print("="*10)

  env = os.environ.copy()

  #r = subprocess.run(cmd, text=True, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env)

  all_lines = []
  with open(live_log_path, "w", encoding="utf-8") as log_f:
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env
    )
    for line in process.stdout:
      print(line, end="")
      log_f.write(line)
      log_f.flush()
      all_lines.append(line)

    returncode = process.wait()

  #log = {"cmd": cmd, "returncode": r.returncode, "stdout": r.stdout, "stderr": r.stderr}

  #with open(log_path, "w", encoding="utf-8") as f:
  #  json.dump(log, f, ensure_ascii=False, indent=2)

  #print("return code: ", r.returncode)

  return out_dir, returncode

In [11]:
for pdf_path in pdf1_paths:
  out_test, returncode = run_mineru(pdf_path)

Running: 
mineru -p /content/drive/MyDrive/프젝랩/(1-7장)산업용기계중대재해사례집_2012년.pdf -o /content/drive/MyDrive/프젝랩/out/(1-7장)산업용기계중대재해사례집_2012년 -m auto -l korean -f false -t true -s 0 -e 5 -t true
return code:  0


In [13]:
for pdf_path in pdf1_paths:
  out_test, returncode = run_mineru(pdf_path)

Running: 
mineru -p /content/drive/MyDrive/프젝랩/(1-7장)산업용기계중대재해사례집_2012년.pdf -o /content/drive/MyDrive/프젝랩/out/(1-7장)산업용기계중대재해사례집_2012년 -m auto -l korean -f false -t true -s 6 -e 11
return code:  0


In [20]:
language_candicate = ['네팔', '라오스', '키르기스스탄', '파키스탄', '동티모르', '몽골', '미얀마', '스리랑카', '방글라데시', '베트남', '우즈베키스탄', '인도네시아', '중국', '캄보디아', '태국', '필리핀', '영어', '몽고', '러시아']

In [26]:
pdf1_split_dir = "/content/drive/MyDrive/프젝랩/PDF1_split"
pdf1_splits = sorted([str(p) for p in Path(pdf1_split_dir).iterdir()])

text_base = Path("/content/drive/MyDrive/프젝랩/PDF1_text")
text_base.mkdir(parents=True, exist_ok=True)

checkpoint_root = Path("/content/drive/MyDrive/프젝랩/PDF1_mineru_checkpoint")
checkpoint_root.mkdir(parents=True, exist_ok=True)

local_root = Path("/content/mineru_work")
local_input_dir = local_root / "input"
local_output_dir = local_root / "output"
local_input_dir.mkdir(parents=True, exist_ok=True)
local_output_dir.mkdir(parents=True, exist_ok=True)

In [27]:
def count_page(pdf_path):
  doc = fitz.open(pdf_path)
  n = doc.page_count
  doc.close()
  return n

def copy2local(pdf_path):
  pdf_path = Path(pdf_path)
  local_path = local_input_dir / pdf_path.stem
  shutil.copy2(pdf_path, local_path)
  return local_path

In [39]:
import os
import json
import time
import subprocess
from pathlib import Path

def run_mineru(
    pdf_path,
    lang="korean",
    out_root=output_dir,
    method="auto",
    table=False,
    formula=False,
    use_gpu=True,
    start_page=None,
    end_page=None,
):
    pdf_path = Path(pdf_path)
    out_dir = Path(out_root) / pdf_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    # 페이지 범위가 있으면 chunk별 폴더 생성
    if start_page is not None and end_page is not None:
        out_dir = out_root / pdf_path.stem / f"p{start_page:04d}_{end_page:04d}"
    else:
        out_dir = out_root / pdf_path.stem

    out_dir.mkdir(parents=True, exist_ok=True)

    done_path = out_dir / "DONE.json"
    failed_path = out_dir / "FAILED.json"
    live_log_path = out_dir / "run_log_live.txt"

    if done_path.exists():
        print(f"[SKIP] already done: {pdf_path.name} {start_page}-{end_page}")
        return out_dir, 0

    if failed_path.exists():
        failed_path.unlink()

    cmd = [
        "mineru",
        "-p", str(pdf_path),
        "-o", str(out_dir),
        "-m", method,
        "-l", lang,
        "-f", str(formula).lower(),
        "-t", str(table).lower(),
    ]

    if start_page is not None:
        cmd += ["-s", str(start_page)]

    if end_page is not None:
        cmd += ["-e", str(end_page)]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    if use_gpu:
        env["CUDA_VISIBLE_DEVICES"] = "0"

    print("=" * 80)
    print("Running:")
    print(" ".join(cmd))
    print("Output:", out_dir)
    print("Live log:", live_log_path)
    print("=" * 80)

    t0 = time.time()

    with open(live_log_path, "w", encoding="utf-8") as log_f:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )

        for line in process.stdout:
            print(line, end="")
            log_f.write(line)
            log_f.flush()

        returncode = process.wait()

    elapsed = time.time() - t0

    md_files = sorted(out_dir.rglob("*.md"))
    json_files = sorted(out_dir.rglob("*.json"))
    image_files = sorted(
        list(out_dir.rglob("*.png")) +
        list(out_dir.rglob("*.jpg")) +
        list(out_dir.rglob("*.jpeg")) +
        list(out_dir.rglob("*.webp"))
    )

    status = {
        "pdf_path": str(pdf_path),
        "cmd": cmd,
        "lang": lang,
        "method": method,
        "table": table,
        "formula": formula,
        "start_page": start_page,
        "end_page": end_page,
        "returncode": returncode,
        "elapsed_sec": elapsed,
        "elapsed_min": elapsed / 60,
        "num_md": len(md_files),
        "num_json": len(json_files),
        "num_images": len(image_files),
        "live_log_path": str(live_log_path),
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    if returncode == 0 and len(md_files) > 0:
        with open(done_path, "w", encoding="utf-8") as f:
            json.dump(status, f, ensure_ascii=False, indent=2)

        print(f"\n[DONE] {pdf_path.name} pages {start_page}-{end_page}")

    else:
        with open(failed_path, "w", encoding="utf-8") as f:
            json.dump(status, f, ensure_ascii=False, indent=2)

        print(f"\n[FAILED] {pdf_path.name} pages {start_page}-{end_page}")

    print("=" * 80)
    print("return code:", returncode)
    print("elapsed min:", round(elapsed / 60, 2))
    print("md:", len(md_files))
    print("json:", len(json_files))
    print("images:", len(image_files))
    print("=" * 80)

    return out_dir, returncode

In [43]:
def run_mineru_by_chunks(
    pdf_path,
    out_root,
    lang="korean",
    chunk_size=32,
    method="auto",
    table=False,
    formula=False,
    use_gpu=True,
):
    pdf_path = Path(pdf_path)
    n_pages = count_page(pdf_path)

    print("#" * 80)
    print("PDF:", pdf_path.name)
    print("pages:", n_pages)
    print("chunk_size:", chunk_size)
    print("#" * 80)

    results = []

    for start in range(0, n_pages, chunk_size):
        end = min(start + chunk_size - 1, n_pages - 1)

        out_dir, code = run_mineru(
            pdf_path=pdf_path,
            lang=lang,
            out_root=out_root,
            method=method,
            table=table,
            formula=formula,
            use_gpu=use_gpu,
            start_page=start,
            end_page=end,
        )

        results.append({
            "pdf_path": str(pdf_path),
            "start_page": start,
            "end_page": end,
            "out_dir": str(out_dir),
            "returncode": code,
        })

        if code != 0:
            print("이 chunk에서 실패:", start, end)
            print("일단 다음 chunk로 넘어가지 않고 중단합니다.")
            break

    merge_mineru_chunks_for_pdf(
        pdf_path=pdf_path,
        out_root=out_root,
        merged_name="MERGED.md"
    )


    return results

In [42]:
from pathlib import Path
import re
import shutil
import fitz

def chunk_sort_key(path: Path):
    """
    p0000_0031, p0032_0063, p0064_0113 같은 폴더를 페이지 순서대로 정렬
    """
    m = re.search(r"p(\d+)_(\d+)", path.name)
    if m:
        return int(m.group(1)), int(m.group(2))
    return 10**9, 10**9


def get_page_count(pdf_path):
    doc = fitz.open(pdf_path)
    n = doc.page_count
    doc.close()
    return n


def get_done_page_ranges(doc_dir):
    """
    doc_dir 안에서 DONE.json이 있는 chunk 폴더들의 page range 수집
    """
    doc_dir = Path(doc_dir)
    ranges = []

    if not doc_dir.exists():
        return ranges

    for chunk_dir in doc_dir.iterdir():
        if not chunk_dir.is_dir():
            continue

        if not (chunk_dir / "DONE.json").exists():
            continue

        m = re.search(r"p(\d+)_(\d+)", chunk_dir.name)
        if not m:
            continue

        start = int(m.group(1))
        end = int(m.group(2))
        ranges.append((start, end, chunk_dir))

    ranges.sort(key=lambda x: (x[0], x[1]))
    return ranges


def is_pdf_fully_done(pdf_path, out_root):
    """
    PDF의 모든 페이지가 DONE chunk로 덮였는지 확인
    """
    pdf_path = Path(pdf_path)
    doc_dir = Path(out_root) / pdf_path.stem
    n_pages = get_page_count(pdf_path)

    done_pages = set()

    for start, end, chunk_dir in get_done_page_ranges(doc_dir):
        done_pages.update(range(start, end + 1))

    return len(done_pages) >= n_pages


def merge_mineru_chunks_for_pdf(pdf_path, out_root, merged_name="MERGED.md"):
    """
    한 PDF의 chunk별 md를 하나의 MERGED.md로 합침.
    이미지도 doc_dir/merged_images로 복사하고 md 안의 이미지 경로를 수정함.
    """
    pdf_path = Path(pdf_path)
    out_root = Path(out_root)

    doc_dir = out_root / pdf_path.stem
    doc_dir.mkdir(parents=True, exist_ok=True)

    ranges = get_done_page_ranges(doc_dir)

    if not ranges:
        print("[MERGE SKIP] DONE chunk가 없습니다:", pdf_path.name)
        return None

    # 모든 페이지가 완료되지 않았으면 merge하지 않음
    if not is_pdf_fully_done(pdf_path, out_root):
        print("[MERGE SKIP] 아직 모든 chunk가 끝나지 않았습니다:", pdf_path.name)
        return None

    merged_images_dir = doc_dir / "merged_images"
    merged_images_dir.mkdir(parents=True, exist_ok=True)

    merged_parts = []

    for start, end, chunk_dir in ranges:
        md_files = sorted([
            p for p in chunk_dir.rglob("*.md")
            if p.name != merged_name and p.name != "MERGED.md"
        ])

        if not md_files:
            print("[WARN] md 없음:", chunk_dir)
            continue

        for md_path in md_files:
            text = md_path.read_text(encoding="utf-8", errors="ignore")

            # MinerU는 보통 md 파일 옆에 images 폴더를 둠
            images_dir = md_path.parent / "images"

            if images_dir.exists():
                image_files = sorted(
                    list(images_dir.glob("*.png")) +
                    list(images_dir.glob("*.jpg")) +
                    list(images_dir.glob("*.jpeg")) +
                    list(images_dir.glob("*.webp"))
                )

                for img_path in image_files:
                    new_img_name = f"{chunk_dir.name}__{img_path.name}"
                    new_img_path = merged_images_dir / new_img_name

                    if not new_img_path.exists():
                        shutil.copy2(img_path, new_img_path)

                    # md 내부 상대경로 보정
                    old_refs = [
                        f"images/{img_path.name}",
                        f"./images/{img_path.name}",
                    ]

                    for old_ref in old_refs:
                        text = text.replace(
                            old_ref,
                            f"merged_images/{new_img_name}"
                        )

            header = (
                f"\n\n"
                f"<!-- CHUNK: {chunk_dir.name} | PAGES: {start}-{end} | "
                f"SOURCE_MD: {md_path.relative_to(doc_dir)} -->"
                f"\n\n"
            )

            merged_parts.append(header + text)

    if not merged_parts:
        print("[MERGE SKIP] 합칠 md가 없습니다:", pdf_path.name)
        return None

    merged_text = "\n".join(merged_parts)
    merged_path = doc_dir / merged_name
    merged_path.write_text(merged_text, encoding="utf-8")

    print("[MERGED]", pdf_path.name)
    print("  saved:", merged_path)
    print("  chunks:", len(ranges))
    print("  length:", len(merged_text))
    print("  images:", merged_images_dir)

    return merged_path

In [ ]:
start_idx = 3

for pdf_path in pdf1_splits[start_idx:]:
    title = Path(pdf_path).stem

    if any(lang in title for lang in language_candicate):
        language = "auto"
    else:
        language = "korean"

    results = run_mineru_by_chunks(
        pdf_path=pdf_path,
        out_root=text_base,
        lang=language,
        chunk_size=50,
        method="auto",
        table=False,
        formula=False,
        use_gpu=True,
    )

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
2026-05-21 19:32:44.362 | INFO     | mineru.utils.engine_utils:get_vlm_engine:34 - Using vllm-async-engine as the inference engine for VLM.
2026-05-21 19:32:48.450674: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 113359.57it/s]
2026-05-21 19:32:55.041 | INFO     | mineru.backend.vlm.utils:enable_custom_logits_processors:50 - compute_capability: 7.5 < 8.0, but vllm version: 0.11.2 >= 0.10.2, enable custom_logits_processors
INFO 05-21 19:32:55 [model.py:631] Resolved architecture: Qwen2VLForConditionalGeneration
WARNING 05-21 19:32:55 [model.py:1921] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torc